In [ ]:
%matplotlib widget

In [ ]:
import numpy as np
import hyperspy.api as hs
import pyxem as pxm
import matplotlib.pyplot as plt

from skimage.filters import threshold_otsu, gaussian, difference_of_gaussians
from skimage.draw import circle_perimeter
from skimage.segmentation import find_boundaries
from skimage.transform import hough_circle, hough_circle_peaks

## Load data

In [ ]:
s = hs.load("data/2.zspy", lazy=True)
s_bs = s.get_direct_beam_position(method="center_of_mass")
s_bs.compute()
s_bs_lp = s_bs.get_linear_plane()
s_bs_corr = s_bs-s_bs_lp

In [ ]:
s_sum = s.sum(axis=(-1, -2))
s_sum.compute()

In [ ]:
fig, ax = plt.subplots()
ax.set_axis_off()

In [ ]:
s_sum_crop = s_sum.inav[50:214, :]

In [ ]:
ax.imshow(s_sum_crop, cmap="grey")

# Gaussian

In [ ]:
s_sum_gauss = gaussian(s_sum_crop.data, 1.5)

In [ ]:
ax.imshow(s_sum_gauss, cmap="grey")

## Otsu

In [ ]:
thresh = threshold_otsu(s_sum_gauss)
s_binary = s_sum_gauss < thresh

In [ ]:
ax.imshow(s_binary, cmap="grey")

## Find Circles

In [ ]:
# Inner vs outer here decides if it takes the dark or the light side of the bounadry
edges = find_boundaries(s_binary, mode="outer")
circle_radius = 8
circle_count = 47

In [ ]:
hough_radii = np.array([circle_radius]*(circle_count-10)+[7]*5+[9]*5)
hough_res = hough_circle(edges, hough_radii)
accums, cx, cy, radii = hough_circle_peaks(
    hough_res,
    hough_radii,
    total_num_peaks=circle_count,
    min_xdistance=int(circle_radius*1.5),
    min_ydistance=int(circle_radius*1.5)
)

In [ ]:
for center_y, center_x, radius in zip(cy, cx, radii):
    circy, circx = circle_perimeter(center_y, center_x, radius, shape=s_binary.shape)
    ax.plot(circx, circy)